In [ ]:
import os
from pathlib import Path

from dotenv import load_dotenv
from crewai import Agent, Task, Crew, Process, LLM
from crewai.tools import tool
from docx import Document
from IPython.display import Markdown, display
from tavily import TavilyClient

In [ ]:
from langsmith.integrations.otel import OtelSpanProcessor
from opentelemetry import trace
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.trace import SpanKind
from opentelemetry.trace.status import Status, StatusCode
from opentelemetry.instrumentation.crewai import CrewAIInstrumentor
from opentelemetry.instrumentation.openai import OpenAIInstrumentor
from wrapt import wrap_function_wrapper

load_dotenv(override=True)

# .env sets OTEL_SDK_DISABLED=true to suppress CrewAI telemetry noise.
# Override before TracerProvider is created or it goes into no-op mode.
os.environ["OTEL_SDK_DISABLED"] = "false"

current_provider = trace.get_tracer_provider()
if isinstance(current_provider, TracerProvider):
    tracer_provider = current_provider
else:
    tracer_provider = TracerProvider()
    trace.set_tracer_provider(tracer_provider)

tracer_provider.add_span_processor(OtelSpanProcessor())

CrewAIInstrumentor().instrument(tracer_provider=tracer_provider)
OpenAIInstrumentor().instrument(tracer_provider=tracer_provider)

# CrewAIInstrumentor only patches the SYNC path:
#   Crew.kickoff, Task.execute_sync, LLM.call
# This notebook uses kickoff_async(), which routes through:
#   Crew.kickoff_async -> Task.execute_async -> LLM.acall
# None of those are patched, so zero spans reach LangSmith.
# Manually wrap the async entry points below.
_tracer = trace.get_tracer("crewai.async", tracer_provider=tracer_provider)

def _wrap_kickoff_async(wrapped, instance, args, kwargs):
    async def _inner():
        with _tracer.start_as_current_span("crewai.workflow", kind=SpanKind.INTERNAL) as span:
            span.set_attribute("gen_ai.system", "crewai")
            try:
                result = await wrapped(*args, **kwargs)
                span.set_status(Status(StatusCode.OK))
                if hasattr(result, "raw"):
                    span.set_attribute("crewai.crew.result", str(result.raw)[:2000])
                return result
            except Exception as e:
                span.set_status(Status(StatusCode.ERROR, str(e)))
                raise
    return _inner()

def _wrap_acall(wrapped, instance, args, kwargs):
    async def _inner():
        model = getattr(instance, "model", "unknown")
        with _tracer.start_as_current_span(f"{model}.llm", kind=SpanKind.CLIENT) as span:
            span.set_attribute("gen_ai.request.model", model)
            try:
                result = await wrapped(*args, **kwargs)
                span.set_status(Status(StatusCode.OK))
                return result
            except Exception as e:
                span.set_status(Status(StatusCode.ERROR, str(e)))
                raise
    return _inner()

wrap_function_wrapper("crewai.crew", "Crew.kickoff_async", _wrap_kickoff_async)
wrap_function_wrapper("crewai.llm", "LLM.acall", _wrap_acall)

print("OTel + LangSmith instrumentation ready.")
print("Project:", os.getenv("LANGSMITH_PROJECT", "default"))

In [ ]:
load_dotenv(override=True)

APIs = {
    "openai_api_key": os.getenv("OPENAI_API_KEY"),
    "groq_api_key": os.getenv("GROQ_API_KEY"), 
    "google_api_key": os.getenv("GOOGLE_API_KEY"),
    "tavily_api_key": os.getenv("TAVILY_API_KEY")
}

# Use .items() to grab both the name string and the secret key value
for name, key_value in APIs.items():
    if key_value:  
        masked_secret = f"{key_value[:8]}..." if len(key_value) > 8 else "Loaded"
        print(f"{name}: loaded (starts with {masked_secret})")
    else:
        print(f"Warning: {name} not found in environment variables")


In [ ]:
llm = LLM(
    model="gemini/gemini-999-fake",
    api_key=APIs["google_api_key"],
    temperature=0.3,
    num_retries=3,
    # is_litellm=True forces CrewAI to skip its native Gemini SDK and route
    # through LiteLLM instead. Without this, gemini/* models go to GeminiCompletion
    # which has no knowledge of fallbacks - they are silently dropped.
    is_litellm=True,
    fallbacks=[
            {
                "model": "openai/gpt-4o-mini",
                "api_key": os.getenv("OPENAI_API_KEY"),
                "temperature": 0.3,
            }
        ],
)

# To prove fallback works: swap the model to "gemini/gemini-999-fake" and run.
# LiteLLM will fail to resolve it and immediately call openai/gpt-4o-mini.
print("Primary: gemini/gemini-1.5-flash | Fallback: openai/gpt-4o-mini")
print(f"LiteLLM path active: {llm.is_litellm}")

In [ ]:
report_inputs = {
    "topic": "Environmental Security in India",
    "audience": "policy analysts, journalists, and informed general readers",
    "tone": "clear, engaging, and natural, never robotic or repetitive",
    "context": "AI infra requirements, climate change, rainfed agri, groundwater extraction, and food security",
    "intro_word_limit": 300
}


intro_writer_agent = Agent(
    role="Introduction Writer",
    goal=(
        "Write an engaging introduction that frames the report's topic, explains why it "
        "matters right now, previews what the rest of the report will cover, and "
        "opens with the compelling quote provided by the web research agent."
    ),
    backstory=(
        "You open long-form reports for a living. Readers decide in the first paragraph "
        "whether to keep reading, so you hook them with a concrete detail or a striking "
        "quote before zooming out to the bigger picture."
    ),
    llm=llm,
    verbose=True,
)

intro_task = Task(
    description=(
        "Using the evidence dossier and the web research output as background in the "
        "context of {context}, write the Introduction section of a report on '{topic}'. "
        "Open with the 'Intro Quote' provided by the web research agent — attribute it "
        "correctly — then explain what the issue is, why it matters right now, and "
        "briefly preview what the rest of the report will cover. "
        "Write in a {tone} tone for {audience}."
    ),
    expected_output=(
        "An Introduction section in Markdown (roughly 3-5 paragraphs), starting with the "
        "heading '## Introduction'. The first paragraph must open with the attributed "
        "quote from the web research output. "
        "Strict limitation: Do not exceed {intro_word_limit} words."
    ),
    agent=intro_writer_agent
)

In [ ]:
crew = Crew(
    agents=[
        intro_writer_agent
    ],
    tasks=[
        intro_task,
    ],
    process=Process.sequential,
    verbose=True,
    cache=False
)

In [ ]:
crew

In [ ]:
print("Starting execution pipeline. This may take a moment as agents cross-examine data components sequentially...")
result = await crew.kickoff_async(inputs=report_inputs)

In [ ]:
print("\n=== SYSTEM PIPELINE COMPLETE ===")
print(result.raw)

In [ ]:
# Extract the token usage metadata dictionary
metrics = result.token_usage
metrics